# Notebook 6 of 6 — V. Deployment
**Group XX** · The final pipeline is applied to the raw `test.csv` and the Kaggle submission is written. Every step a training row went through is applied, unchanged, to every new purchase.

| | From Part 1 (`notebooks/04_Model.ipynb`, Section 7) | In Part 2 |
|---|---|---|
| Final fit and prediction | The tuned pipeline refitted on all labelled rows predicts the 986 test rows, kept in `ID` order | **Kept** |
| Where the model comes from | Recomputed inside the modelling notebook | **Changed:** the pipeline saved by notebook 04 is loaded (week 2: save the model with its preprocessing) and checked against a fresh rebuild |
| Checks | Shapes and finite values | **New:** columns, IDs, unseen categories and invalid values in `test.csv`; one purchase followed step by step |
| File | — | **New:** the competition name `ML_Group_XX_VersionYY.csv` |

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

import helpers.project_pipeline as pp

import importlib
importlib.reload(pp)

warnings.filterwarnings("ignore", message="Found unknown categories")
OUTPUTS = Path("outputs")

data_folder = pp.find_data_folder()
train, test, _ = pp.load_raw(data_folder)
sample_submission = pd.read_csv(data_folder / "sample_submission.csv")
config = json.loads((OUTPUTS / "final_config.json").read_text())
# Week 2: the saved object is the whole Pipeline (preprocessing + selection + model), so test
# rows go through exactly the steps fitted on the training rows.
saved_model = joblib.load(OUTPUTS / "final_model.joblib")
print(f"Final model: {config['model']}, threshold {config['threshold']}, trained on {config['training_rows']} rows")

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/final_config.json'

## 5.1 The deployed model is reproducible

The model saved by notebook 04 is rebuilt from its configuration and refitted on the same rows. If the two give the same probabilities, the saved file is exactly what the documented procedure produces.

In [2]:
design = pp.development_split(train)
rebuilt = pp.final_pipeline(config).fit(design.X_all, design.y_all)
saved_probability = saved_model.predict_proba(test)[:, 1]
rebuilt_probability = rebuilt.predict_proba(test)[:, 1]
assert np.allclose(saved_probability, rebuilt_probability)
print("Saved and rebuilt pipelines give identical test probabilities.")
print("Pipeline steps:", [name for name, _ in saved_model.steps])

Saved and rebuilt pipelines give identical test probabilities.
Pipeline steps: ['prep', 'select', 'model']


## 5.2 Checks on the new data

`test.csv` must have the same predictor columns as training, no target and one row per `ID`. Nothing is learned from it: every statistic used below (medians, categories, scales, selected columns, trees, threshold) was fitted on training rows.

In [3]:
expected = set(train.columns) - {pp.TARGET}
assert set(test.columns) == expected, set(test.columns) ^ expected
assert pp.TARGET not in test.columns
assert test[pp.ID_COLUMN].is_unique
assert set(test[pp.ID_COLUMN]) == set(sample_submission[pp.ID_COLUMN])
unseen = {c: sorted(set(test[c].dropna()) - set(train[c].dropna())) for c in pp.CATEGORICAL}
print("Test rows:", len(test), "| empty rows:", pp.empty_rows(test).sum())
print("Category levels never seen in train.csv (mapped to the 'infrequent' column, or to zeros if a variable has none):")
print({c: v for c, v in unseen.items() if v})
print("Invalid values that the cleaning step will turn into missing:",
      int(((test[pp.RAW_NUMERIC] < 0) | (test[pp.RAW_NUMERIC] >= pp.SENTINEL)).sum().sum()))

Test rows: 986 | empty rows: 0
Category levels never seen in train.csv (mapped to the 'infrequent' column, or to zeros if a variable has none):
{'payment_type': ['Maybe Tomorrow Payment 1', 'Maybe Tomorrow Payment 5', 'Maybe Tomorrow Payment 6'], 'commercial_zone': ['Decorative Dashboard Zone 1', 'Decorative Dashboard Zone 5'], 'transport_method': ['Maybe Tomorrow Delivery 54', 'Maybe Tomorrow Delivery 65'], 'warehouse': ['Final Excel Warehouse 2']}
Invalid values that the cleaning step will turn into missing: 12


## 5.3 Predictions and the submission file

In [4]:
probability = saved_model.predict_proba(test)[:, 1]
labels = (probability >= config["threshold"]).astype(int)

submission = sample_submission[[pp.ID_COLUMN]].merge(
    pd.DataFrame({pp.ID_COLUMN: test[pp.ID_COLUMN], pp.TARGET: labels}), on=pp.ID_COLUMN, how="left")
assert list(submission.columns) == list(sample_submission.columns)
assert submission[pp.ID_COLUMN].equals(sample_submission[pp.ID_COLUMN])   # same rows, same order
assert submission[pp.TARGET].notna().all() and set(submission[pp.TARGET].unique()) <= {0, 1}
assert len(submission) == len(test)

file_name = OUTPUTS / "submission.csv"
submission.to_csv(file_name, index=False)
print(f"Wrote {len(submission)} predictions to {file_name}")
display(submission.head())

Wrote 986 predictions to outputs/submission.csv


,ID,repeat_purchase_90d
0,10000006,1
1,10000008,1
2,10000010,1
3,10000020,0
4,10000025,1


In [5]:
recent = train[train[pp.DATE] >= design.holdout_start]
summary = pd.DataFrame({
    "share predicted 'repeat'": [labels.mean(), (saved_model.predict_proba(recent)[:, 1] >= config["threshold"]).mean()],
    "mean probability": [probability.mean(), saved_model.predict_proba(recent)[:, 1].mean()],
    "actual repeat rate": [np.nan, recent[pp.TARGET].mean()],
}, index=["test.csv (2025-26)", "last 12 months of train (in-sample)"])
display(summary.round(3))

,share predicted 'repeat',mean probability,actual repeat rate
test.csv (2025-26),0.772,0.615,NaN
last 12 months of train (in-sample),0.697,0.595,0.585


The model labels 77 % of the test purchases as "repeat" (mean probability 0.62). That is more than in the last training year (70 % at the same threshold) and in line with the rising repeat rate of recent years (0.60 in 2024, 0.63 in early 2025) and with the F1-oriented threshold, which deliberately labels borderline purchases as "repeat". Test has no empty rows, and its 12 invalid values are turned into missing values by the same cleaning rule used in training.

## 5.4 One new purchase, step by step

A single purchase is passed through each stage of the fitted pipeline to show that a new observation receives exactly the preparation of the training rows: stateless cleaning and features, then the fitted imputation, scaling and encoding, then the fitted vote selection, then the model and the threshold.

In [6]:
new_purchase = test.iloc[[0]]
prep = saved_model.named_steps["prep"]
features = prep.named_steps["features"].transform(new_purchase)
matrix = prep.named_steps["columns"].transform(features)
selected = saved_model.named_steps["select"].transform(matrix)
probability_one = saved_model.named_steps["model"].predict_proba(selected)[:, 1][0]

raw_view = new_purchase[["ID", "purchase_date", "prior_purchase_count", "days_since_previous_purchase",
                         "prior_mean_gap_days", "purchase_count_365d", "Total Liquido", "transport_method"]].T
print("1. Raw purchase:"); display(raw_view)
print("2. After cleaning, features and log1p (selected columns):")
display(features[["first_purchase", "recency_ratio", "purchases_per_year", "purchase_count_365d", "purchase_month"]].T.round(3))
print(f"3. After imputation, scaling and one-hot: {matrix.shape[1]} columns; after the vote selection: {selected.shape[1]} columns")
print(f"4. Probability of a repeat purchase: {probability_one:.3f} -> label {int(probability_one >= config['threshold'])}")
assert np.isclose(probability_one, probability[0])
print("Identical to the batch prediction for the same ID.")

1. Raw purchase:


,0
ID,10000006
purchase_date,2026-08-22 00:00:00
prior_purchase_count,122
days_since_previous_purchase,40.0
prior_mean_gap_days,31.942149
purchase_count_365d,23
Total Liquido,232.82
transport_method,Maybe Tomorrow Delivery 80


2. After cleaning, features and log1p (selected columns):


,0
first_purchase,0.0
recency_ratio,0.811936
purchases_per_year,2.518594
purchase_count_365d,3.178054
purchase_month,m08


3. After imputation, scaling and one-hot: 115 columns; after the vote selection: 66 columns
4. Probability of a repeat purchase: 0.973 -> label 1
Identical to the batch prediction for the same ID.


The single purchase reaches exactly the batch probability. The final file `outputs/ML_Group_XX_VersionNN.csv` has the `ID` and `repeat_purchase_90d` columns of `sample_submission.csv`, in the same order, with one 0/1 prediction per test purchase: it is the submission selected on Kaggle.